### Model training

In [41]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
# set display columns to max so that we can view all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
warnings.filterwarnings("ignore")

# Optional: nicer display for pandas in notebooks
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

# Update this path if your CSV is in a different location within the project
csv_path = Path("data/clean/US_Accidents_March23_NY_Clean.csv")

if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found at {csv_path}. Please place the file there or change the path in this cell.")

# Read CSV (use low_memory=False to avoid dtype warnings for large CSVs)
df = pd.read_csv(csv_path, low_memory=False)
print("Loaded:", csv_path, "-> shape:", df.shape)

# Show the first 5 rows so you can inspect column names and sample values
df.head()

Loaded: data\clean\US_Accidents_March23_NY_Clean.csv -> shape: (307740, 47)


,ID,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),Description,Street,City,County,State,Timezone,Weather_Timestamp,Temperature(F),Wind_Chill(F),Visibility(mi),Pressure(in),Humidity(%),Weather_Condition,Wind_Direction,Wind_Speed(mph),Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Hour,Weekday,Month,Day,Day_of_week,Is_weekend,Duration_min,Weather_raw,Weather_Short,Season,Severity_Class,Street_encoded
0,A-194257,3,2016-11-30 16:25:41,2016-11-30 17:10:30,41.139324,-74.169876,0.0100,Right lane blocked due to accident on I-87 Southbound around Exit 15A NY-17.,Route 17,Sloatsburg,Rockland,NY,US/Eastern,2016-11-30 16:53:00,53.1,51.0,1.0,29.81,93.0,Overcast,East,11.5,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.816667,overcast,Cloudy/Clear,Autumn,Major,118
1,A-194258,3,2016-11-30 15:40:06,2016-11-30 17:09:40,41.118748,-74.159744,1.4925,Accident on I-87 Southbound between Exit 14B County Hwy-89 Airmont Rd and Exit 15 I-287.,I-287 S,Suffern,Rockland,NY,US/Eastern,2016-11-30 15:53:00,53.1,51.0,1.0,29.83,93.0,Overcast,ENE,8.1,0,0,0,0,0,0,0,0,0,0,0,0,0,15,Wednesday,November,30,2,0,89.566667,overcast,Cloudy/Clear,Autumn,Major,141
2,A-194259,3,2016-11-30 15:26:18,2016-11-30 16:56:02,41.014034,-73.717239,1.1700,Accident on I-287 Cross Westchester Expy Eastbound between Exit 9A I-684 and Exit 9S-N I-684.,Hutchinson River Pkwy S,White Plains,Westchester,NY,US/Eastern,2016-11-30 15:56:00,52.0,51.0,2.0,29.81,93.0,Light Rain,East,17.3,0,0,0,0,1,0,0,0,0,0,0,0,0,15,Wednesday,November,30,2,0,89.733333,light rain,Rain/Precip,Autumn,Major,1682
3,A-194260,3,2016-11-30 16:14:11,2016-11-30 16:59:00,41.390369,-73.597969,0.0100,Right hand shoulder blocked due to accident on I-84 Westbound at Exits 20 20N 20S I-684.,I-84 E,Brewster,Putnam,NY,US/Eastern,2016-11-30 15:53:00,52.0,51.0,3.0,29.86,89.0,Rain,East,10.4,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.816667,rain,Rain/Precip,Autumn,Major,1895
4,A-194261,3,2016-11-30 16:17:57,2016-11-30 17:02:44,41.390369,-73.597969,0.0100,Right hand shoulder blocked due to accident on I-84 Eastbound at Exits 20 20N 20S I-684.,I-84 E,Brewster,Putnam,NY,US/Eastern,2016-11-30 15:53:00,52.0,51.0,3.0,29.86,89.0,Rain,East,10.4,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.783333,rain,Rain/Precip,Autumn,Major,1895


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307740 entries, 0 to 307739
Data columns (total 47 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   ID                 307740 non-null  object 
 1   Severity           307740 non-null  int64  
 2   Start_Time         307740 non-null  object 
 3   End_Time           307740 non-null  object 
 4   Start_Lat          307740 non-null  float64
 5   Start_Lng          307740 non-null  float64
 6   Distance(mi)       307740 non-null  float64
 7   Description        307739 non-null  object 
 8   Street             307740 non-null  object 
 9   City               307740 non-null  object 
 10  County             307740 non-null  object 
 11  State              307740 non-null  object 
 12  Timezone           307740 non-null  object 
 13  Weather_Timestamp  307740 non-null  object 
 14  Temperature(F)     307740 non-null  float64
 15  Wind_Chill(F)      307740 non-null  float64
 16  Vi

In [43]:
target_col = "Severity_Class" 
infra_cols = [
    'Amenity','Bump','Crossing','Give_Way','Junction','No_Exit','Railway',
    'Roundabout','Station','Stop','Traffic_Calming','Traffic_Signal','Turning_Loop'
] 

numeric_cols = ['Distance(mi)','Temperature(F)','Visibility(mi)',
                        'Humidity(%)','Pressure(in)','Wind_Speed(mph)','Hour','Duration_min']

category_cols = ["Street_encoded", "County","Weather_Condition","Wind_Direction","Weather_Short","Season"]

feature_cols = numeric_cols + category_cols + infra_cols
# Combined categorical columns (infrastructure flags are categorical / indicator features)
print("Categorical columns ", category_cols + infra_cols)
print("Numeric columns ", numeric_cols )

Categorical columns  ['Street_encoded', 'County', 'Weather_Condition', 'Wind_Direction', 'Weather_Short', 'Season', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop']
Numeric columns  ['Distance(mi)', 'Temperature(F)', 'Visibility(mi)', 'Humidity(%)', 'Pressure(in)', 'Wind_Speed(mph)', 'Hour', 'Duration_min']


In [44]:
df.isna().sum()

ID                   0
Severity             0
Start_Time           0
End_Time             0
Start_Lat            0
Start_Lng            0
Distance(mi)         0
Description          1
Street               0
City                 0
County               0
State                0
Timezone             0
Weather_Timestamp    0
Temperature(F)       0
Wind_Chill(F)        0
Visibility(mi)       0
Pressure(in)         0
Humidity(%)          0
Weather_Condition    0
Wind_Direction       0
Wind_Speed(mph)      0
Amenity              0
Bump                 0
Crossing             0
Give_Way             0
Junction             0
No_Exit              0
Railway              0
Roundabout           0
Station              0
Stop                 0
Traffic_Calming      0
Traffic_Signal       0
Turning_Loop         0
Hour                 0
Weekday              0
Month                0
Day                  0
Day_of_week          0
Is_weekend           0
Duration_min         0
Weather_raw          0
Weather_Sho

In [45]:
df.head(5)

,ID,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),Description,Street,City,County,State,Timezone,Weather_Timestamp,Temperature(F),Wind_Chill(F),Visibility(mi),Pressure(in),Humidity(%),Weather_Condition,Wind_Direction,Wind_Speed(mph),Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Hour,Weekday,Month,Day,Day_of_week,Is_weekend,Duration_min,Weather_raw,Weather_Short,Season,Severity_Class,Street_encoded
0,A-194257,3,2016-11-30 16:25:41,2016-11-30 17:10:30,41.139324,-74.169876,0.0100,Right lane blocked due to accident on I-87 Southbound around Exit 15A NY-17.,Route 17,Sloatsburg,Rockland,NY,US/Eastern,2016-11-30 16:53:00,53.1,51.0,1.0,29.81,93.0,Overcast,East,11.5,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.816667,overcast,Cloudy/Clear,Autumn,Major,118
1,A-194258,3,2016-11-30 15:40:06,2016-11-30 17:09:40,41.118748,-74.159744,1.4925,Accident on I-87 Southbound between Exit 14B County Hwy-89 Airmont Rd and Exit 15 I-287.,I-287 S,Suffern,Rockland,NY,US/Eastern,2016-11-30 15:53:00,53.1,51.0,1.0,29.83,93.0,Overcast,ENE,8.1,0,0,0,0,0,0,0,0,0,0,0,0,0,15,Wednesday,November,30,2,0,89.566667,overcast,Cloudy/Clear,Autumn,Major,141
2,A-194259,3,2016-11-30 15:26:18,2016-11-30 16:56:02,41.014034,-73.717239,1.1700,Accident on I-287 Cross Westchester Expy Eastbound between Exit 9A I-684 and Exit 9S-N I-684.,Hutchinson River Pkwy S,White Plains,Westchester,NY,US/Eastern,2016-11-30 15:56:00,52.0,51.0,2.0,29.81,93.0,Light Rain,East,17.3,0,0,0,0,1,0,0,0,0,0,0,0,0,15,Wednesday,November,30,2,0,89.733333,light rain,Rain/Precip,Autumn,Major,1682
3,A-194260,3,2016-11-30 16:14:11,2016-11-30 16:59:00,41.390369,-73.597969,0.0100,Right hand shoulder blocked due to accident on I-84 Westbound at Exits 20 20N 20S I-684.,I-84 E,Brewster,Putnam,NY,US/Eastern,2016-11-30 15:53:00,52.0,51.0,3.0,29.86,89.0,Rain,East,10.4,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.816667,rain,Rain/Precip,Autumn,Major,1895
4,A-194261,3,2016-11-30 16:17:57,2016-11-30 17:02:44,41.390369,-73.597969,0.0100,Right hand shoulder blocked due to accident on I-84 Eastbound at Exits 20 20N 20S I-684.,I-84 E,Brewster,Putnam,NY,US/Eastern,2016-11-30 15:53:00,52.0,51.0,3.0,29.86,89.0,Rain,East,10.4,0,0,0,0,0,0,0,0,0,0,0,0,0,16,Wednesday,November,30,2,0,44.783333,rain,Rain/Precip,Autumn,Major,1895


In [46]:
# Work on a copy to be safe
df_clean = df.copy()


# Infrastructure flags -> integer (0/1)
try: df['Amenity'] = df['Amenity'].astype('Int8')
except: print("Amenity cast failed")
try: df['Bump'] = df['Bump'].astype('Int8')
except: print("Bump cast failed")
try: df['Crossing'] = df['Crossing'].astype('Int8')
except: print("Crossing cast failed")
try: df['Give_Way'] = df['Give_Way'].astype('Int8')
except: print("Give_Way cast failed")
try: df['Junction'] = df['Junction'].astype('Int8')
except: print("Junction cast failed")
try: df['No_Exit'] = df['No_Exit'].astype('Int8')
except: print("No_Exit cast failed")
try: df['Railway'] = df['Railway'].astype('Int8')
except: print("Railway cast failed")
try: df['Roundabout'] = df['Roundabout'].astype('Int8')
except: print("Roundabout cast failed")
try: df['Station'] = df['Station'].astype('Int8')
except: print("Station cast failed")
try: df['Stop'] = df['Stop'].astype('Int8')
except: print("Stop cast failed")
try: df['Traffic_Calming'] = df['Traffic_Calming'].astype('Int8')
except: print("Traffic_Calming cast failed")
try: df['Traffic_Signal'] = df['Traffic_Signal'].astype('Int8')
except: print("Traffic_Signal cast failed")
try: df['Turning_Loop'] = df['Turning_Loop'].astype('Int8')
except: print("Turning_Loop cast failed")

# Numeric columns -> float
try: df['Distance(mi)'] = df['Distance(mi)'].astype('float64')
except: print("Distance(mi) cast failed")
try: df['Temperature(F)'] = df['Temperature(F)'].astype('float64')
except: print("Temperature(F) cast failed")
try: df['Visibility(mi)'] = df['Visibility(mi)'].astype('float64')
except: print("Visibility(mi) cast failed")
try: df['Humidity(%)'] = df['Humidity(%)'].astype('float64')
except: print("Humidity(%) cast failed")
try: df['Pressure(in)'] = df['Pressure(in)'].astype('float64')
except: print("Pressure(in) cast failed")
try: df['Wind_Speed(mph)'] = df['Wind_Speed(mph)'].astype('float64')
except: print("Wind_Speed(mph) cast failed")
try: df['Hour'] = df['Hour'].astype('Int64')
except: 
    try: df['Hour'] = df['Hour'].astype('float64')
    except: print("Hour cast failed")
try: df['Duration_min'] = df['Duration_min'].astype('float64')
except: print("Duration_min cast failed")

# Categorical columns -> category
try: df['Street_encoded'] = df['Street_encoded'].astype('category')
except: print("Street_encoded cast failed")
try: df['City'] = df['City'].astype('category')
except: print("City cast failed")
try: df['County'] = df['County'].astype('category')
except: print("County cast failed")
try: df['Weather_Condition'] = df['Weather_Condition'].astype('category')
except: print("Weather_Condition cast failed")
try: df['Wind_Direction'] = df['Wind_Direction'].astype('category')
except: print("Wind_Direction cast failed")
try: df['Weather_Short'] = df['Weather_Short'].astype('category')
except: print("Weather_Short cast failed")
try: df['Season'] = df['Season'].astype('category')
except: print("Season cast failed")

# Description -> string (text)
if 'Description' in df.columns:
    try: df['Description'] = df['Description'].astype('string')
    except: print("Description cast failed")
        
# Start_Time and End_Time -> datetime
# uses errors='coerce' to convert invalid formats to NaT while preserving the column
if 'Start_Time' in df.columns:
    try:
        df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
        print("Start_Time cast to datetime64[ns]")
    except Exception:
        print("Start_Time cast failed")
else:
    print("Start_Time: MISSING")

if 'End_Time' in df.columns:
    try:
        df['End_Time'] = pd.to_datetime(df['End_Time'], errors='coerce')
        print("End_Time cast to datetime64[ns]")
    except Exception:
        print("End_Time cast failed")
else:
    print("End_Time: MISSING")

# Target -> category (replace target_col with your target variable name)
try:
    df[target_col] = df[target_col].astype('category')
except Exception:
    print(f"{target_col} cast to category failed")

Start_Time cast to datetime64[ns]
End_Time cast to datetime64[ns]


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307740 entries, 0 to 307739
Data columns (total 47 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   ID                 307740 non-null  object        
 1   Severity           307740 non-null  int64         
 2   Start_Time         307740 non-null  datetime64[ns]
 3   End_Time           307740 non-null  datetime64[ns]
 4   Start_Lat          307740 non-null  float64       
 5   Start_Lng          307740 non-null  float64       
 6   Distance(mi)       307740 non-null  float64       
 7   Description        307739 non-null  string        
 8   Street             307740 non-null  object        
 9   City               307740 non-null  category      
 10  County             307740 non-null  category      
 11  State              307740 non-null  object        
 12  Timezone           307740 non-null  object        
 13  Weather_Timestamp  307740 non-null  object  

In [48]:
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report

In [49]:
# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --------------------------------------------------------------------------
# 1) Train/test split
# - Stratify by target so class proportions are preserved in train/test
# - This prevents accidental optimistic evaluation when classes are imbalanced
# --------------------------------------------------------------------------
X = df[feature_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)


In [50]:
# --------------------------------------------------------------------------
# 2) Preprocessing pipelines
# - Numeric: median imputation + standard scaling (robust and works for most models)
# - Categorical & infra flags: most-frequent imputation + one-hot encoding
# - Keep preprocessing inside a pipeline to avoid train-test leakage and to make CV safe
# --------------------------------------------------------------------------

num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),   # fill numeric missing values
    ('scale', StandardScaler())                      # scale numeric features for linear models
])

# Treat infra flags as categorical/indicator features; OneHotEncoder will create 0/1 columns
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent'))
    ,   # fill missing categories
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse=False))  # safe for unseen categories
])

cat_cols = category_cols + infra_cols
df[cat_cols] = df[cat_cols].astype("string")
preprocessor = ColumnTransformer([
    ('num', num_pipe, numeric_cols),
    ('cat', cat_pipe, category_cols )
], remainder='drop')

In [51]:
df[cat_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307740 entries, 0 to 307739
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Street_encoded     307740 non-null  string
 1   County             307740 non-null  string
 2   Weather_Condition  307740 non-null  string
 3   Wind_Direction     307740 non-null  string
 4   Weather_Short      307740 non-null  string
 5   Season             307740 non-null  string
 6   Amenity            307740 non-null  string
 7   Bump               307740 non-null  string
 8   Crossing           307740 non-null  string
 9   Give_Way           307740 non-null  string
 10  Junction           307740 non-null  string
 11  No_Exit            307740 non-null  string
 12  Railway            307740 non-null  string
 13  Roundabout         307740 non-null  string
 14  Station            307740 non-null  string
 15  Stop               307740 non-null  string
 16  Traffic_Calming    3

In [52]:
# --------------------------------------------------------------------------
# 3) Baseline models
# - LogisticRegression: interpretable baseline with class_weight="balanced"
# - DecisionTree: simple nonlinear baseline
# - RandomForest: ensemble baseline capturing interactions and non-linearities
# Each model is wrapped in a pipeline that includes the preprocessor
# --------------------------------------------------------------------------

lr_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', multi_class='multinomial', max_iter=1000, random_state=RANDOM_SEED))
])

dt_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_SEED))
])

rf_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1))
])

In [53]:
# Fit baselines and print evaluation (weighted F1 is primary metric)
print("Fitting baseline models...")
for name, pipe in [('LogisticRegression', lr_pipeline), ('DecisionTree', dt_pipeline), ('RandomForest', rf_pipeline)]:
    pipe.fit(X_train, y_train)                       # train model
    preds = pipe.predict(X_test)                     # predict on test set
    print(f"\n=== {name} ===")
    print("Weighted F1:", f1_score(y_test, preds, average='weighted'))  # primary metric
    print(classification_report(y_test, preds))     # precision/recall per class;
    ; 

Fitting baseline models...

=== LogisticRegression ===
Weighted F1: 0.705558173311028
              precision    recall  f1-score   support

       Major       0.52      0.69      0.59     15861
       Minor       0.03      0.81      0.05       248
    Moderate       0.90      0.64      0.75     45439

    accuracy                           0.66     61548
   macro avg       0.48      0.71      0.46     61548
weighted avg       0.80      0.66      0.71     61548


=== DecisionTree ===
Weighted F1: 0.8294897695737788
              precision    recall  f1-score   support

       Major       0.67      0.70      0.68     15861
       Minor       0.32      0.45      0.37       248
    Moderate       0.89      0.87      0.88     45439

    accuracy                           0.83     61548
   macro avg       0.63      0.67      0.65     61548
weighted avg       0.83      0.83      0.83     61548


=== RandomForest ===
Weighted F1: 0.8618777442468631
              precision    recall  f1-score 

In [54]:
# -----------------------
# XGBoost pipeline (preprocessing -> XGB classifier)
# - We use sklearn wrapper XGBClassifier so it fits into Pipeline
# -----------------------
xgb_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', xgb.XGBClassifier(objective='multi:softprob', random_state=RANDOM_SEED, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1))
])

In [55]:
# -----------------------
# Option: train with SMOTE (example)
# - Use imblearn Pipeline so SMOTE is applied only to training folds
# -----------------------
use_smote = False  # set True to try oversampling
if use_smote:
    xgb_smote_pipeline = ImbPipeline([
        ('preproc', preprocessor),
        ('smote', SMOTE(random_state=RANDOM_SEED)),
        ('clf', xgb.XGBClassifier(objective='multi:softprob', random_state=RANDOM_SEED, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1))
    ])

In [56]:
# -----------------------
# Quick baseline fit for XGBoost (default params)
# -----------------------
# print("\nFitting XGBoost baseline (default params)...")

# xgb_pipeline.fit(X_train, y_train)
# xgb_preds = xgb_pipeline.predict(X_test)
# print("XGBoost baseline weighted F1:", f1_score(y_test, xgb_preds, average='weighted'))
# print(classification_report(y_test, xgb_preds))

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

print("\nFitting XGBoost baseline (default params)...")

# encode labels (fit only on training labels)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# fit and predict with encoded labels
xgb_pipeline.fit(X_train, y_train_enc)
xgb_preds_enc = xgb_pipeline.predict(X_test)

# decode predictions back to original string labels for readable reports
xgb_preds = le.inverse_transform(xgb_preds_enc)
y_test_labels = le.inverse_transform(y_test_enc)

print("XGBoost baseline weighted F1:", f1_score(y_test_labels, xgb_preds, average='weighted'))
print(classification_report(y_test_labels, xgb_preds))


Fitting XGBoost baseline (default params)...
XGBoost baseline weighted F1: 0.8606063861614679
              precision    recall  f1-score   support

       Major       0.82      0.63      0.72     15861
       Minor       0.67      0.17      0.27       248
    Moderate       0.88      0.95      0.91     45439

    accuracy                           0.87     61548
   macro avg       0.79      0.59      0.63     61548
weighted avg       0.86      0.87      0.86     61548



In [59]:
# -----------------------
# Hyperparameter tuning (RandomizedSearchCV)
# - StratifiedKFold preserves class proportions across folds
# - Optimize for weighted F1 (primary metric)
# - Keep search compact for speed; expand ranges for full tuning
# -----------------------
param_dist = {
    'clf__max_depth': [4, 6, 8],
    'clf__learning_rate': [0.01, 0.03, 0.1],
    'clf__n_estimators': [100, 300, 600],
    'clf__subsample': [0.6, 0.8, 1.0],
    'clf__colsample_bytree': [0.6, 0.8, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
rand_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist,
    n_iter=5,                 # reduce for speed; increase for thorough search
    scoring='f1_weighted',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1,
    error_score="raise"   # <- raise exception so you get the real traceback
)

print("\nRunning RandomizedSearchCV for XGBoost...")
t0 = time.time()
rand_search.fit(X_train, y_train)
t1 = time.time()
print(f"RandomizedSearchCV done in {t1 - t0:.1f}s")
best_xgb = rand_search.best_estimator_
print("Best params:", rand_search.best_params_)
print("Best CV weighted F1:", rand_search.best_score_)


Running RandomizedSearchCV for XGBoost...
Fitting 5 folds for each of 5 candidates, totalling 25 fits


MemoryError: Unable to allocate 813. MiB for an array with shape (196953, 541) and data type float64

In [22]:
#  !pip install catboost


   ---------------------------------------- 0.0/102.5 MB ? eta -:--:--
   - -------------------------------------- 3.9/102.5 MB 29.4 MB/s eta 0:00:04
   ---- ----------------------------------- 10.7/102.5 MB 28.0 MB/s eta 0:00:04
   ------ --------------------------------- 17.0/102.5 MB 29.9 MB/s eta 0:00:03
   --------- ------------------------------ 23.3/102.5 MB 30.2 MB/s eta 0:00:03
   ---------- ----------------------------- 27.3/102.5 MB 27.9 MB/s eta 0:00:03
   ------------ --------------------------- 31.5/102.5 MB 25.9 MB/s eta 0:00:03
   ------------- -------------------------- 34.6/102.5 MB 24.1 MB/s eta 0:00:03
   -------------- ------------------------- 37.5/102.5 MB 23.1 MB/s eta 0:00:03
   ---------------- ----------------------- 41.4/102.5 MB 22.5 MB/s eta 0:00:03
   ----------------- ---------------------- 45.1/102.5 MB 21.9 MB/s eta 0:00:03
   ------------------- -------------------- 49.0/102.5 MB 21.5 MB/s eta 0:00:03
   -------------------- ------------------- 52.2/

In [24]:
# %pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 15.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [25]:
#  pip install xgboost

   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   - -------------------------------------- 2.6/72.0 MB 16.9 MB/s eta 0:00:05
   ----- ---------------------------------- 10.0/72.0 MB 27.0 MB/s eta 0:00:03
   -------- ------------------------------- 16.0/72.0 MB 28.0 MB/s eta 0:00:03
   ------------ --------------------------- 22.0/72.0 MB 27.9 MB/s eta 0:00:02
   --------------- ------------------------ 27.8/72.0 MB 28.4 MB/s eta 0:00:02
   ------------------ --------------------- 34.1/72.0 MB 28.5 MB/s eta 0:00:02
   --------------------- ------------------ 38.8/72.0 MB 27.4 MB/s eta 0:00:02
   ----------------------- ---------------- 42.5/72.0 MB 26.2 MB/s eta 0:00:02
   -------------------------- ------------- 46.9/72.0 MB 25.3 MB/s eta 0:00:01
   --------------------------- ------------ 50.1/72.0 MB 24.3 MB/s eta 0:00:01
   ----------------------------- ---------- 53.0/72.0 MB 23.4 MB/s eta 0:00:01
   ------------------------------- -------- 56.1/72.0 MB 22.5 

In [32]:
# -----------------------
# XGBoost pipeline (preprocessing -> XGB classifier)
# - We use sklearn wrapper XGBClassifier so it fits into Pipeline
# -----------------------
xgb_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', xgb.XGBClassifier(objective='multi:softprob', random_state=RANDOM_SEED, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1))
])

In [28]:
# Benchmark CatBoost, LightGBM, XGBoost
# Requirements: pip install scikit-learn imbalanced-learn lightgbm xgboost catboost
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, recall_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# learners
from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# -------------------------
# USER: set df and target_col
# -------------------------
# df = pd.read_csv("...") 
# target_col = "severity_label"
# assert 'df' in globals(), "Load your DataFrame into `df`"
# assert 'target_col' in globals(), "Set target_col to your target column name"

# -------------------------
# Columns (adjust if needed)
# -------------------------
infra_cols = ['Amenity','Bump','Crossing','Give_Way','Junction','No_Exit','Railway',
             'Roundabout','Station','Stop','Traffic_Calming','Traffic_Signal','Turning_Loop']
numeric_cols = ['Distance(mi)','Temperature(F)','Visibility(mi)','Humidity(%)',
                'Pressure(in)','Wind_Speed(mph)','Hour','Duration_min']
category_cols = ["Street","City","County","Weather_Condition","Wind_Direction","Weather_Short","Season"]
# Keep only columns present in df
numeric_cols = [c for c in numeric_cols if c in df.columns]
category_cols = [c for c in category_cols if c in df.columns]
infra_cols = [c for c in infra_cols if c in df.columns]

# Feature matrix and labels
feature_cols = numeric_cols + category_cols + infra_cols
X = df[feature_cols]
y = df[target_col].astype('category')  # ensure categorical labels



In [29]:
# If you have a known severe label, set it; otherwise choose last category as heuristic
severe_label = list(y.cat.categories)[-1]

# -------------------------
# Preprocessors
# -------------------------
# Numeric pipeline: median impute + scale
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

# Categorical pipeline for LightGBM/XGBoost: most-frequent impute + one-hot
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

# ColumnTransformer for LightGBM/XGBoost pipelines
preproc_ohe = ColumnTransformer([
    ('num', num_pipe, numeric_cols),
    ('cat', cat_pipe, category_cols + infra_cols)
], remainder='drop')

# For CatBoost we will pass raw DataFrame after simple imputation for numeric columns,
# and pass categorical feature names/indices directly (no OHE).
def catboost_prepare(df_sub):
    # numeric impute only (CatBoost handles category strings natively)
    df_num = df_sub[numeric_cols].copy()
    df_num = df_num.fillna(df_num.median())
    df_cat = df_sub[category_cols + infra_cols].copy()
    return pd.concat([df_num, df_cat], axis=1)



In [30]:
# -------------------------
# Cross-validation setup
# -------------------------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

# Short hyperparameter lists for quick comparisons (expand if you want fuller tuning)
cat_params = [
    {'iterations': 200, 'depth': 6, 'learning_rate': 0.05},
    {'iterations': 500, 'depth': 8, 'learning_rate': 0.03}
]
lgb_params = [
    {'num_leaves':31, 'n_estimators':200, 'learning_rate':0.05},
    {'num_leaves':63, 'n_estimators':400, 'learning_rate':0.03}
]
xgb_params = [
    {'max_depth':6, 'n_estimators':200, 'learning_rate':0.05, 'use_label_encoder':False},
    {'max_depth':8, 'n_estimators':400, 'learning_rate':0.03, 'use_label_encoder':False}
]

# -------------------------
# Helper: evaluate one model across folds
# -------------------------
def evaluate_model(name, model_builder, use_catboost=False, use_smote=False, params_list=None):
    fold_scores = []
    fold_severe_recall = []
    fold_times = []
    params_used = params_list if params_list is not None else [None]
    # We'll pick the best set from params_list by CV weighted F1
    best_cv_score = -np.inf
    best_summary = None

    for params in params_used:
        fold_scores = []
        fold_severe_recall = []
        fold_times = []
        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            X_tr = X.iloc[train_idx].copy()
            X_val = X.iloc[val_idx].copy()
            y_tr = y.iloc[train_idx].copy()
            y_val = y.iloc[val_idx].copy()

            if use_catboost:
                X_tr_pre = catboost_prepare(X_tr)
                X_val_pre = catboost_prepare(X_val)
                cat_features = list(range(len(numeric_cols), X_tr_pre.shape[1]))  # indices of categorical cols
                model = model_builder(params, cat_features)
                start = time.time()
                model.fit(X_tr_pre, y_tr, verbose=0)
                elapsed = time.time() - start
                preds = model.predict(X_val_pre)
            else:
                # Preprocess with OHE pipeline
                X_tr_pre = preproc_ohe.fit_transform(X_tr)
                X_val_pre = preproc_ohe.transform(X_val)
                if use_smote:
                    sm = SMOTE(random_state=RANDOM_SEED)
                    X_tr_pre, y_tr = sm.fit_resample(X_tr_pre, y_tr)
                model = model_builder(params, None)
                start = time.time()
                model.fit(X_tr_pre, y_tr)
                elapsed = time.time() - start
                preds = model.predict(X_val_pre)

            # metrics
            w_f1 = f1_score(y_val, preds, average='weighted')
            rec_sev = recall_score(y_val, preds, labels=[severe_label], average='macro', zero_division=0)
            fold_scores.append(w_f1)
            fold_severe_recall.append(rec_sev)
            fold_times.append(elapsed)

        avg_score = np.mean(fold_scores)
        if avg_score > best_cv_score:
            best_cv_score = avg_score
            best_summary = {
                'params': params,
                'cv_weighted_f1_mean': avg_score,
                'cv_weighted_f1_std': np.std(fold_scores),
                'cv_severe_recall_mean': np.mean(fold_severe_recall),
                'cv_time_mean_sec': np.mean(fold_times)
            }

    # return best summary across param choices
    print(f"\n{name} best CV summary:")
    print(best_summary)
    return best_summary



In [31]:
# -------------------------
# Run benchmarks
# -------------------------
results = {}
# CatBoost (native categorical handling)
results['CatBoost'] = evaluate_model('CatBoost', build_catboost, use_catboost=True, use_smote=False, params_list=cat_params)

# LightGBM (with OHE preprocessing)
results['LightGBM'] = evaluate_model('LightGBM', build_lgb, use_catboost=False, use_smote=False, params_list=lgb_params)

# XGBoost (with OHE preprocessing)
results['XGBoost'] = evaluate_model('XGBoost', build_xgb, use_catboost=False, use_smote=False, params_list=xgb_params)

# -------------------------
# Quick fairness check on test set using the best model chosen above
# (we'll pick the model with highest cv_weighted_f1_mean)
# -------------------------
best_model_name = max(results.items(), key=lambda kv: kv[1]['cv_weighted_f1_mean'])[0]
print("\nBest model by CV weighted F1:", best_model_name)

# Train chosen best model on full train (80%) and evaluate per-group on held-out 20% test
# Recreate train/test split deterministically
from sklearn.model_selection import train_test_split
X_full = df[feature_cols]
y_full = df[target_col].astype('category')
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, stratify=y_full, random_state=RANDOM_SEED)

if best_model_name == 'CatBoost':
    X_tr = catboost_prepare(X_train)
    X_te = catboost_prepare(X_test)
    cat_features = list(range(len(numeric_cols), X_tr.shape[1]))
    best_model = build_catboost(results['CatBoost']['params'], cat_features)
    best_model.fit(X_tr, y_train)
    preds = best_model.predict(X_te)
else:
    X_tr = preproc_ohe.fit_transform(X_train)
    X_te = preproc_ohe.transform(X_test)
    if best_model_name == 'LightGBM':
        best_model = build_lgb(results['LightGBM']['params'], None)
    else:
        best_model = build_xgb(results['XGBoost']['params'], None)
    best_model.fit(X_tr, y_train)
    preds = best_model.predict(X_te)

# Per-group metrics (County, Season)
print("\nPer-group metrics on holdout test set (groups with >=30 samples):")
preds_series = pd.Series(preds, index=y_test.index)
for group_col in ['County', 'Season']:
    if group_col not in X_test.columns:
        continue
    print(f"\nGroup: {group_col}")
    for grp, idxs in X_test.groupby(group_col).groups.items():
        idxs = list(idxs)
        if len(idxs) < 30:
            continue
        y_true_grp = y_test.loc[idxs]
        y_pred_grp = preds_series.loc[idxs]
        f1_grp = f1_score(y_true_grp, y_pred_grp, average='weighted')
        rec_sev_grp = recall_score(y_true_grp, y_pred_grp, labels=[severe_label], average='macro', zero_division=0)
        print(f"{grp}: n={len(idxs)} weighted_f1={f1_grp:.3f} recall_severe={rec_sev_grp:.3f}")

# Final classification report on test set
print("\nOverall classification report (best model on test set):")
print(classification_report(y_test, preds))

KeyboardInterrupt: 

In [ ]:
# -------------------------
# Model builders
# -------------------------
def build_catboost(params, cat_features):
    # params is a dict or None
    p = {} if params is None else params.copy()
    # Class weights to address imbalance — CatBoost accepts class_weights param (list)
    # We compute simple class weights inversely proportional to class frequency
    classes = y.cat.categories
    counts = y.value_counts().reindex(classes).fillna(0).astype(float)
    inv = (1.0 / counts).replace([np.inf], 0.0).fillna(0.0)
    weights = (inv / inv.sum()).tolist()
    p.setdefault('class_weights', weights)
    p.setdefault('random_state', RANDOM_SEED)
    p.setdefault('verbose', 0)
    # Create model
    model = CatBoostClassifier(**p)
    # We pass cat_features at fit time in evaluate_model
    # but CatBoostClassifier allows cat_features in fit call
    # We'll define a small wrapper that supports .fit(X, y) where X is DataFrame
    class CBWrapper:
        def __init__(self, model, cat_features):
            self.model = model
            self.cat_features = cat_features
        def fit(self, X, y, **kwargs):
            # X is DataFrame
            self.model.fit(X, y, cat_features=self.cat_features, **kwargs)
            return self
        def predict(self, X):
            return self.model.predict(X)
    return CBWrapper(model, cat_features)

def build_lgb(params, _):
    p = {} if params is None else params.copy()
    # set objective for multiclass if needed, else let LightGBM infer
    p.setdefault('random_state', RANDOM_SEED)
    p.setdefault('n_jobs', -1)
    return lgb.LGBMClassifier(**p)

def build_xgb(params, _):
    p = {} if params is None else params.copy()
    p.setdefault('random_state', RANDOM_SEED)
    p.setdefault('n_jobs', -1)
    # xgboost sklearn API may warn; ensure use_label_encoder handled in params
    return xgb.XGBClassifier(**p)



In [ ]:
search_string = 'Accord'

# Find columns where any cell value contains the search string
columns_with_string = [col for col in df.columns if df[col].astype(str).str.contains(search_string, case=False).any()]

print(columns_with_string)
# Output

In [ ]:
from sklearn.exceptions import NotFittedError
try:
    # fit only the preprocessor on training data to locate transformer error early
    preprocessor.fit(X_train)
    print("preprocessor.fit succeeded")
except Exception as e:
    print("preprocessor.fit failed with:")
    print(e)
    # Inspect X_train.sample for likely culprits
    print("\nSample rows that might cause problem:")
    print(X_train.sample(10))

In [ ]:
# show dtypes for columns used by the pipeline
print("Numeric columns:", numeric_cols)
print("Category columns:", category_cols)
print("Infra columns:", infra_cols)

print("\nDataFrame dtypes (columns of interest):")
print(df[numeric_cols + category_cols + infra_cols].dtypes)

# find which numeric column contains non-numeric values (samples)
for col in numeric_cols:
    if col in df.columns:
        # try to coerce a small sample; any non-convertible value will become NaN
        sample = df[col].dropna().astype(str).head(200)
        non_numeric = sample[~sample.apply(lambda v: v.replace('.', '', 1).lstrip('-').isdigit())]
        if len(non_numeric)>0:
            print(f"\nColumn {col} has non-numeric examples (showing up to 10):")
            print(non_numeric.unique()[:10])